In [1]:
import pandas as pd
import numpy as np
import os

# Setting pandas to show all columns so we can inspect our data properly
pd.set_option('display.max_columns', None)
print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
# Define the path to our raw data
path = '../data/raw/'

# Load the core tables we need for our analysis
print("Loading datasets...")
orders = pd.read_csv(path + 'olist_orders_dataset.csv')
items = pd.read_csv(path + 'olist_order_items_dataset.csv')
reviews = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
customers = pd.read_csv(path + 'olist_customers_dataset.csv')

print("Data loaded successfully!")
print(f"Total Orders: {len(orders)}")


Loading datasets...
Data loaded successfully!
Total Orders: 99441


In [3]:
# 1. Convert string dates into actual Datetime objects
date_columns = ['order_purchase_timestamp', 'order_approved_at', 
                'order_delivered_carrier_date', 'order_delivered_customer_date', 
                'order_estimated_delivery_date']

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# 2. Calculate the Delivery Delay (in days)
# If the number is positive, it was delayed. If negative, it arrived early.
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

# 3. Create a simple Flag for "Late Delivery" (1 = Late, 0 = On Time)
orders['is_late'] = orders['delivery_delay_days'].apply(lambda x: 1 if x > 0 else 0)

print("Dates cleaned and delivery metrics calculated!")


Dates cleaned and delivery metrics calculated!


In [4]:
# Merge Orders with Items (Using order_id)
master_df = pd.merge(orders, items, on='order_id', how='left')

# Merge with Customer Geolocation Data (Using customer_id)
master_df = pd.merge(master_df, customers, on='customer_id', how='left')

# Merge with Reviews (Using order_id)
# We take the most recent review if there are multiple per order
reviews_clean = reviews.drop_duplicates(subset=['order_id'], keep='last')
master_df = pd.merge(master_df, reviews_clean, on='order_id', how='left')

print(f"Master Dataset created! Shape: {master_df.shape}")


Master Dataset created! Shape: (113425, 26)


In [5]:
# Drop rows where we don't have delivery dates or prices (cleaning up anomalies)
master_df_clean = master_df.dropna(subset=['price', 'order_delivered_customer_date'])

# Save the final optimized dataset to our 'processed' folder
output_path = '../data/processed/Master_Analytical_Table.csv'
master_df_clean.to_csv(output_path, index=False)

print(f"Success! Master dataset saved to: {output_path}")


Success! Master dataset saved to: ../data/processed/Master_Analytical_Table.csv
